In [1]:
vocab = {
    "<PAD>": 0,
    "<START>": 1,
    "I": 2,
    "love": 3,
    "AI": 4,
    "and": 5,
    "ML": 6,
    "<END>": 7
}

In [3]:
import tensorflow as tf
from tensorflow.keras import layers

In [4]:
vocab_size = 8
embed_dim = 16
num_heads = 2
ff_dim = 32

In [5]:
token_embedding = layers.Embedding(
    input_dim=vocab_size,
    output_dim=embed_dim
)

In [7]:
input_tokens = tf.constant([[2, 3]])

embedded = token_embedding(input_tokens)

print("Shape:", embedded.shape)
print(embedded.numpy())

Shape: (1, 2, 16)
[[[-2.1447098e-02 -2.5359616e-03  2.7832773e-02  3.5444722e-03
   -2.0916963e-02 -5.5193901e-05 -4.0035594e-02 -2.1546138e-02
    2.5889289e-02 -2.3695339e-02  3.4034241e-02  2.7730618e-02
    4.3437481e-03  1.3722960e-02 -3.3935010e-02  3.8472142e-02]
  [ 4.5458898e-03 -7.6740496e-03 -3.4959972e-02  2.6628207e-02
    7.7234283e-03 -3.4865119e-02  2.9459622e-02 -3.8554780e-03
    4.7152583e-02  2.9208791e-02 -3.1430293e-02  4.8940871e-02
    2.5310982e-02  2.5611971e-02 -3.3858076e-02  4.6447281e-02]]]


In [8]:
max_sequence_length = 10

position_embedding = layers.Embedding(
    input_dim=max_sequence_length,
    output_dim=embed_dim
)

In [9]:
positions = tf.range(start=0, limit=2)

print("Positions:", positions.numpy())

Positions: [0 1]


In [10]:
position_vectors = position_embedding(positions)

print("Position embedding shape:", position_vectors.shape)
print(position_vectors.numpy())

Position embedding shape: (2, 16)
[[ 0.0440565   0.0060281  -0.00747435 -0.03989746  0.03008009 -0.00992347
  -0.00814127  0.04229598 -0.01811226  0.00051783  0.03453321  0.0009579
  -0.01392825  0.0207049   0.02000361  0.03146389]
 [ 0.02476919 -0.00772285 -0.04156534 -0.01246082  0.00392022  0.04349722
   0.02836088 -0.02421221 -0.022155   -0.01480816 -0.04737836 -0.0056737
   0.03875634  0.00604658 -0.02547228 -0.02877187]]


In [11]:
position_vectors = tf.expand_dims(position_vectors, axis=0)

combined_embedding = embedded + position_vectors

print("Combined shape:", combined_embedding.shape)
print(combined_embedding.numpy())

Combined shape: (1, 2, 16)
[[[ 0.0226094   0.00349214  0.02035842 -0.03635299  0.00916313
   -0.00997866 -0.04817686  0.02074984  0.00777702 -0.0231775
    0.06856745  0.02868851 -0.0095845   0.03442786 -0.0139314
    0.06993603]
  [ 0.02931508 -0.0153969  -0.07652531  0.01416739  0.01164365
    0.0086321   0.0578205  -0.02806768  0.02499758  0.01440063
   -0.07880865  0.04326718  0.06406732  0.03165855 -0.05933036
    0.01767541]]]


In [12]:
attention_layer = layers.MultiHeadAttention(
    num_heads=2,
    key_dim=embed_dim
)

In [13]:
attention_output = attention_layer(
    query=combined_embedding,
    key=combined_embedding,
    value=combined_embedding,
    use_causal_mask=True
)

print("Attention output shape:", attention_output.shape)
print(attention_output.numpy())

Attention output shape: (1, 2, 16)
[[[ 0.0349428   0.01293939  0.02276997 -0.02591927 -0.04965634
   -0.0068349  -0.01835795 -0.03875529  0.03914486  0.00632381
   -0.02424749  0.02064743 -0.04198977  0.00343554  0.01897179
   -0.02331936]
  [ 0.02955472  0.00518239  0.0356452  -0.03009463 -0.01092663
    0.01526411  0.01800598 -0.0368783   0.00229498 -0.01655157
   -0.01083712  0.01174111 -0.01960614  0.02614678 -0.01784896
    0.00774533]]]


In [14]:
attention_output, attention_scores = attention_layer(
    query=combined_embedding,
    key=combined_embedding,
    value=combined_embedding,
    use_causal_mask=True,
    return_attention_scores=True
)

print("Attention output shape:", attention_output.shape)
print("Attention scores shape:", attention_scores.shape)
print(attention_scores.numpy())

Attention output shape: (1, 2, 16)
Attention scores shape: (1, 2, 2, 2)
[[[[1.         0.        ]
   [0.50016534 0.49983463]]

  [[1.         0.        ]
   [0.49983695 0.5001631 ]]]]


In [15]:
norm1 = layers.LayerNormalization()

x = norm1(combined_embedding + attention_output)

print("After Residual + LayerNorm:")
print(x.numpy())

After Residual + LayerNorm:
[[[ 0.99920964  0.22228652  0.72669005 -1.2647225  -0.8532345
   -0.4058376  -1.345258   -0.4283567   0.7983632  -0.40659592
    0.74920297  0.8439739  -1.0625972   0.62721467  0.00706518
    0.79259646]
  [ 0.9272268  -0.24602485 -0.76681525 -0.34304333 -0.06037577
    0.33327395  1.2152002  -1.1755235   0.39095384 -0.10908192
   -1.5949973   0.861647    0.6825267   0.909149   -1.3832808
    0.35916495]]]


In [16]:
ffn = tf.keras.Sequential([
    layers.Dense(32, activation="relu"),
    layers.Dense(16)
])

In [19]:
ffn_output = ffn(x)

print("FFN output shape:", ffn_output.shape)
print(ffn_output.numpy)

FFN output shape: (1, 2, 16)
<bound method _EagerTensorBase.numpy of <tf.Tensor: shape=(1, 2, 16), dtype=float32, numpy=
array([[[ 1.1899948 , -0.9029727 , -0.7367249 , -0.36316797,
          0.7872669 ,  0.02289784,  0.7562965 ,  0.07383534,
          1.2277677 , -0.51247835,  0.9625779 , -0.5806542 ,
          0.11071655, -0.6218376 ,  0.64888346, -1.5644596 ],
        [ 0.75045466, -0.16910651, -0.6721178 ,  0.45896053,
          0.35990372,  0.89131457, -0.1734462 ,  0.6167638 ,
          0.38428968,  0.07332408, -0.39408526,  0.3252654 ,
         -0.05233122, -0.8853471 , -0.32072076, -0.82664657]]],
      dtype=float32)>>


In [20]:
norm2 = layers.LayerNormalization()

final_output = norm2(x + ffn_output)

print("Final Transformer block output shape:")
print(final_output.shape)

print(final_output.numpy())

Final Transformer block output shape:
(1, 2, 16)
[[[ 2.0117824e+00 -6.6355270e-01 -3.8366169e-02 -1.5465441e+00
   -9.0507135e-02 -3.8599110e-01 -5.7804620e-01 -3.5949922e-01
    1.8597639e+00 -8.8578004e-01  1.5667242e+00  2.1645711e-01
   -9.1636229e-01 -2.3999060e-02  5.8246899e-01 -7.4854857e-01]
  [ 1.5798930e+00 -4.1821352e-01 -1.3956850e+00  8.8803433e-02
    2.6410520e-01  1.1473039e+00  9.7274315e-01 -5.5534226e-01
    7.1829313e-01 -5.6007847e-02 -1.9209384e+00  1.1113328e+00
    5.7980901e-01  8.5659698e-04 -1.6487582e+00 -4.6819475e-01]]]


In [21]:
vocab_size = 8

In [22]:
lm_head = layers.Dense(vocab_size)

In [23]:
logits = lm_head(final_output)

print("Logits shape:", logits.shape)
print(logits.numpy())

Logits shape: (1, 2, 8)
[[[ 1.0301985  -1.3836644   0.8097104   0.14228177  0.38799447
   -0.40440124  0.950168   -1.7382323 ]
  [ 0.62017775 -0.32936612  0.7058715   1.5333363  -1.7045766
   -0.25008395 -2.0192583  -0.01151311]]]


In [24]:
probabilities = tf.nn.softmax(logits, axis=-1)

print("Probabilities:")
print(probabilities.numpy())

Probabilities:
[[[0.2467126  0.02207314 0.19789489 0.10152522 0.12980329 0.05876959
   0.22773749 0.01548379]
  [0.16424906 0.06355084 0.17894486 0.40933853 0.01606479 0.06879441
   0.01172763 0.08732991]]]


In [41]:
predicted_tokens = tf.argmax(
    probabilities,
    axis=-1
)

print("Predicted token IDs:")
print(predicted_tokens.numpy())

Predicted token IDs:
[[0 3]]


In [42]:
inputs = tf.constant([[2, 3]])
targets = tf.constant([[3, 4]])

print("Inputs:", inputs.numpy())
print("Targets:", targets.numpy())

Inputs: [[2 3]]
Targets: [[3 4]]


In [43]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

loss = loss_fn(
    targets,
    logits
)

print("Loss:", loss.numpy())

Loss: 0.9082571


In [44]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.01
)

In [45]:
optimizer.apply_gradients(
    zip(gradients, trainable_variables)
)

print("Weights updated successfully!")

Weights updated successfully!


In [46]:
with tf.GradientTape() as tape:

    # Forward pass
    embedded = token_embedding(inputs)

    position_vectors = position_embedding(
        tf.range(start=0, limit=2)
    )

    position_vectors = tf.expand_dims(
        position_vectors,
        axis=0
    )

    combined_embedding = embedded + position_vectors

    # Causal self-attention
    attention_output = attention_layer(
        query=combined_embedding,
        key=combined_embedding,
        value=combined_embedding,
        use_causal_mask=True
    )

    # Residual + LayerNorm
    x = norm1(
        combined_embedding + attention_output
    )

    # FFN
    ffn_output = ffn(x)

    # Residual + LayerNorm
    final_output = norm2(
        x + ffn_output
    )

    # LM Head
    logits = lm_head(final_output)

    # Loss
    loss = loss_fn(
        targets,
        logits
    )

print("Loss:", loss.numpy())

Loss: 0.35331744


In [47]:
trainable_variables = (
    token_embedding.trainable_variables
    + position_embedding.trainable_variables
    + attention_layer.trainable_variables
    + norm1.trainable_variables
    + ffn.trainable_variables
    + norm2.trainable_variables
    + lm_head.trainable_variables
)

gradients = tape.gradient(
    loss,
    trainable_variables
)

print("Number of trainable variables:", len(trainable_variables))
print("Number of gradients:", len(gradients))

Number of trainable variables: 20
Number of gradients: 20


In [48]:
for step in range(20):

    with tf.GradientTape() as tape:

        # Embedding
        embedded = token_embedding(inputs)

        # Position embedding
        position_vectors = position_embedding(
            tf.range(start=0, limit=2)
        )
        position_vectors = tf.expand_dims(
            position_vectors,
            axis=0
        )

        combined_embedding = (
            embedded + position_vectors
        )

        # Causal self-attention
        attention_output = attention_layer(
            query=combined_embedding,
            key=combined_embedding,
            value=combined_embedding,
            use_causal_mask=True
        )

        # Residual + LayerNorm
        x = norm1(
            combined_embedding + attention_output
        )

        # FFN
        ffn_output = ffn(x)

        # Residual + LayerNorm
        final_output = norm2(
            x + ffn_output
        )

        # LM Head
        logits = lm_head(final_output)

        # Loss
        loss = loss_fn(
            targets,
            logits
        )

    # Gradients
    gradients = tape.gradient(
        loss,
        trainable_variables
    )

    # Update weights
    optimizer.apply_gradients(
        zip(gradients, trainable_variables)
    )

    print(
        f"Step {step + 1}, Loss: {loss.numpy():.4f}"
    )

Step 1, Loss: 0.3533
Step 2, Loss: 0.1235
Step 3, Loss: 0.0652
Step 4, Loss: 0.0430
Step 5, Loss: 0.0325
Step 6, Loss: 0.0266
Step 7, Loss: 0.0222
Step 8, Loss: 0.0185
Step 9, Loss: 0.0154
Step 10, Loss: 0.0128
Step 11, Loss: 0.0107
Step 12, Loss: 0.0090
Step 13, Loss: 0.0077
Step 14, Loss: 0.0066
Step 15, Loss: 0.0057
Step 16, Loss: 0.0050
Step 17, Loss: 0.0044
Step 18, Loss: 0.0040
Step 19, Loss: 0.0036
Step 20, Loss: 0.0032


In [49]:
probabilities = tf.nn.softmax(logits, axis=-1)

predicted_tokens = tf.argmax(
    probabilities,
    axis=-1
)

print("Predicted token IDs:")
print(predicted_tokens.numpy())

Predicted token IDs:
[[3 4]]


In [50]:
print("Probabilities:")
print(probabilities.numpy())

Probabilities:
[[[5.8618170e-04 1.9823312e-04 6.5764197e-04 9.9637306e-01 8.8985823e-04
   8.9446880e-04 1.6939486e-04 2.3118671e-04]
  [4.8620193e-04 1.9413116e-04 2.2428179e-04 5.7855528e-04 9.9715602e-01
   2.8772463e-04 3.3931975e-04 7.3378813e-04]]]
